# Duke Bites — Data Prep

Full Preprocessing pipeline for cleaning up and loading data, generating embeddings, and saving generated embeddings.

In [1]:
import sys, os
sys.path.append("../src")

print(sys.path)

from config import MENU_CSV, LOCATIONS_CSV, EMBEDDINGS_PKL, EMBEDDING_MODEL
from preprocessing import load_and_clean, merge_data, build_item_combo, build_chunks
from embeddings import generate_embeddings, save_embeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print('Imports OK.')

['/opt/anaconda3/envs/duke-dining/lib/python310.zip', '/opt/anaconda3/envs/duke-dining/lib/python3.10', '/opt/anaconda3/envs/duke-dining/lib/python3.10/lib-dynload', '', '/opt/anaconda3/envs/duke-dining/lib/python3.10/site-packages', '../src']
Imports OK.


In [2]:
# Load & clean datasets
menu_df, loc_df = load_and_clean(MENU_CSV, LOCATIONS_CSV)
print(f'Menu: {menu_df.shape} | Locations: {loc_df.shape}')
print('Null counts (menu):')
print(menu_df.isnull().sum())
print()
print('Meal periods:', menu_df['meal_period'].value_counts().to_dict())

Menu: (394, 10) | Locations: (16, 11)
Null counts (menu):
item_id              0
location_id          0
name                 0
meal_period          0
description          0
generated_tags       0
name_clean           0
description_clean    0
meal_period_clean    0
tags_list            0
dtype: int64

Meal periods: {'lunch, dinner': 239, 'Drink': 27, 'drink': 24, 'lunch, dinner, snack': 23, 'breakfast': 19, 'dinner': 17, 'dessert': 12, 'snack, dessert': 9, 'breakfast, dessert, lunch': 6, 'breakfast, snack, dessert': 5, 'Breakfast': 3, 'breakfast, lunch': 3, 'snack': 2, 'breakfast, snack': 2, 'lunch, snack': 1, 'breakfast, snack ': 1, 'lunch': 1}


In [3]:
# Merge + build item with location information
merged_df = merge_data(menu_df, loc_df)
merged_df['text_blob'] = merged_df.apply(build_item_combo, axis=1)
print(f'Merged: {merged_df.shape}')
print('\nSample item:')
print(merged_df['text_blob'].iloc[0])

Merged: (394, 17)

Sample item:
bacon, egg, and cheese sandwich is a breakfast item at beyu blue coffee, a cafe dining spot. Description: breakfast sandwich with bacon, egg, and cheese. Tags: breakfast sandwich dairy egg. Venue tags: coffee drinks breakfast grab-and-go quick. Hours: 7 am - 7 pm (mon - fri); 9 am - 5 pm (sat - sun).


In [4]:
# Chunking - build chunking for RAG
chunks_df = build_chunks(merged_df)
print(f'{len(chunks_df)} chunks from {len(merged_df)} items')
print('\nSample chunk:')
print(chunks_df['text_blob'].iloc[0])

47 chunks from 394 items

Sample chunk:
Beyu Blue Coffee (cafe) serves the following Breakfast items:
  - Bacon, Egg, and Cheese Sandwich: breakfast sandwich with bacon, egg, and cheese (tags: breakfast,sandwich,dairy,egg)
  - Egg and Cheese Sandwich: breakfast sandwich with egg and cheese (tags: breakfast,sandwich,eggs,dairy)
  - Sausage, Egg, and Cheese Sandwich: breakfast sandwich with sausage, egg, and cheese (tags: breakfast,sandwich,eggs,dairy)
Hours: 7 am - 7 pm (mon - fri); 9 am - 5 pm (sat - sun)


In [5]:
# Generate embeddings — item level
print('Generating item embeddings...')
item_embeddings = generate_embeddings(merged_df['text_blob'].tolist(), EMBEDDING_MODEL)
print(f'Item embeddings: {item_embeddings.shape}')

# Generate embeddings — chunk level
print('Generating chunk embeddings...')
chunk_embeddings = generate_embeddings(chunks_df['text_blob'].tolist(), EMBEDDING_MODEL)
print(f'Chunk embeddings: {chunk_embeddings.shape}')

Generating item embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Item embeddings: (394, 384)
Generating chunk embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Chunk embeddings: (47, 384)


In [6]:
# Save embeddings
item_meta_cols = [c for c in ['item_id','location_id','name_item','name_location',
                              'meal_period','description_clean','generated_tags',
                              'cuisine_clean','hours','location','text_blob']
                 if c in merged_df.columns]

save_embeddings(
    EMBEDDINGS_PKL,
    item_embeddings, merged_df[item_meta_cols].copy(),
    chunk_embeddings, chunks_df,
    EMBEDDING_MODEL
)

Saved embeddings to /Users/kaylatom/Desktop/DUKE STUFF/CS 372/cs372_final_project/data/embeddings.pkl


In [7]:
# Test Queries
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMBEDDING_MODEL)

test_queries = [
    'I want something spicy for dinner',
    'healthy vegan breakfast',
    'something with toppings',
]

for q in test_queries:
    qvec   = model.encode([q])
    scores = cosine_similarity(qvec, item_embeddings)[0]
    top3   = np.argsort(scores)[::-1][:3]
    print(f"\nQuery: '{q}'")
    for idx in top3:
        row = merged_df.iloc[idx]
        print(f"  {row['name_item']} @ {row['name_location']}  (score: {scores[idx]:.3f})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Query: 'I want something spicy for dinner'
  Spicy Il Forno @ Il Forno  (score: 0.631)
  Spicy Miso Ramen @ Ginger + Soy  (score: 0.600)
  Spicy Tuba Yubu @ Gyotaku  (score: 0.576)

Query: 'healthy vegan breakfast'
  Yogurt and Oatmeal Bar Topppings @ Sprout  (score: 0.697)
  Vegetarian Omelet Combo @ The Skillet  (score: 0.659)
  Avocado Toast @ Sprout  (score: 0.635)

Query: 'something with toppings'
  Toppings Options @ Sazon  (score: 0.507)
  Yogurt and Oatmeal Bar Topppings @ Sprout  (score: 0.418)
  Build Your Own Pizza @ Gothic Grill   (score: 0.403)
